In [1]:
import sys
sys.path.append('../../../')
sys.path.append('../../')

from file_management import get_files_dir,check_save_file
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

In [2]:
import pandas as pd

In [3]:
mapped_to_BIGG = pd.read_json('mapped_to_BIGG.json')
mapped_to_BIGG = mapped_to_BIGG.loc[:,['Gene', 'Paper_ID', 'Chosen_Reaction_ID','Chosen_XREF_to_Matching_ECs','Classification']]

In [4]:
original_df = pd.read_json('Test_normalised_classifed.json')
original_df = original_df.loc[:,['Title', 'Organism']]
original_df.head()

,Title,Organism
0,Increased production of zeaxanthin and other p...,[Synechocystis PCC]
1,Expression of Alcaligenes eutrophus flavohemop...,[Escherichia coli]
2,Environmental biotechnology.,None
3,Altered regulation of pyruvate kinase or co-ov...,[Escherichia coli]
4,Cloning and characterization of the Yarrowia l...,"[Saccharomyces cerevisiae, Yarrowia lipolytica]"


In [5]:
original_df = pd.read_json('Test_normalised_classifed.json')


In [6]:
original_df.head()

,Title,Organism,Genes,Gene_Source,Modifications,EC_Modifications,EC_Modifications_Normalized,EC_Modifications_Classified,EC_Modifications_Classified_min,EC_Modifications_Classified_min_relv
0,Increased production of zeaxanthin and other p...,[Synechocystis PCC],"[[crtB, P37294, syn:slr1255;, 2.5.1.32;], [crt...",abstract,"[ipi -> expression, crtP -> was introduced, cr...","{'ipi': {'EC_numbers': [], 'Modifications': ['...","{'ipi': {'EC_numbers': [], 'Normalized_Modific...","{'ipi': {'EC_numbers': [], 'Classified_Modific...",None,None
1,Expression of Alcaligenes eutrophus flavohemop...,[Escherichia coli],None,not_found,None,{},{},{},None,None
2,Environmental biotechnology.,None,None,not_found,None,{},{},{},None,None
3,Altered regulation of pyruvate kinase or co-ov...,[Escherichia coli],None,not_found,[pyk -> overexpression],"{'pyk': {'EC_numbers': ['2.7.1.40'], 'Modifica...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Normaliz...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi...","{'pyk': {'EC_numbers': ['2.7.1.40'], 'Classifi..."
4,Cloning and characterization of the Yarrowia l...,"[Saccharomyces cerevisiae, Yarrowia lipolytica]","[[SQS1, A6ZRL6_P53866A6ZRL6_P53866, sce:YNL224...",title,None,{},{},{},None,None


In [8]:
mapped_to_BIGG.head()

,Gene,Paper_ID,Chosen_Reaction_ID,Chosen_XREF_to_Matching_ECs,Classification
0,AAE,8268,[MNXR190571],{'ACS': ['6.2.1.1']},Positive
1,AAT1,13075,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...","{'ALATA_L': ['2.6.1.1'], 'ASPTA': ['2.6.1.1'],...",Negative
2,AAT2,4115,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...","{'ALATA_L': ['2.6.1.1'], 'ASPTA': ['2.6.1.1'],...",Positive
3,AAT2,7146,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...","{'ALATA_L': ['2.6.1.1'], 'ASPTA': ['2.6.1.1'],...",Negative
4,AAT2,7717,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...","{'ALATA_L': ['2.6.1.1'], 'ASPTA': ['2.6.1.1'],...",Negative


In [9]:

rows = []
for _, row in mapped_to_BIGG.iterrows():
    xref_dict = row["Chosen_XREF_to_Matching_ECs"] or {}
    for bigg, ecs in xref_dict.items():
        rows.append({
            "Gene": row["Gene"],
            "Paper_ID": row["Paper_ID"],
            "Classification": row["Classification"],

            "Chosen_Reaction_ID": row["Chosen_Reaction_ID"],
            "Chosen_BIGG": bigg,
            "EC_numbers": set(ecs)  # keep the list intact
        })

df_exploded = pd.DataFrame(rows)


In [10]:
df_exploded.head()

,Gene,Paper_ID,Classification,Chosen_Reaction_ID,Chosen_BIGG,EC_numbers
0,AAE,8268,Positive,[MNXR190571],ACS,{6.2.1.1}
1,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ALATA_L,{2.6.1.1}
2,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ASPTA,{2.6.1.1}
3,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",TYRTA,{2.6.1.1}
4,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",LEUTAi,{2.6.1.1}


In [12]:
grouped = df_exploded.groupby(['Chosen_BIGG'])


In [13]:
grouped = df_exploded.groupby('Chosen_BIGG')

genes = grouped.Gene.apply(set)
# Combine EC_numbers into one flattened set
ec_numbers = grouped["EC_numbers"].apply(
    lambda x: set().union(*[e if isinstance(e, (set, list)) else [e] for e in x])
)


# Other group 

In [14]:

# Step 1: create a normalized key for grouping (string form of sorted ECs)
df_exploded["EC_key"] = df_exploded["EC_numbers"].apply(lambda s: ",".join(sorted(s)))

# Step 2: assign a group index per reaction based on EC_key
df_exploded["group_index"] = (
    df_exploded.groupby("Chosen_BIGG")["EC_key"]
    .transform(lambda x: pd.factorize(x)[0] + 1)  # gives 1, 2, 3, ... within each Chosen_BIGG
)

# Step 3: build the group name
df_exploded["Reaction_Group"] = df_exploded["Chosen_BIGG"] + "_" + df_exploded["group_index"].astype(str)

df_exploded.head()

,Gene,Paper_ID,Classification,Chosen_Reaction_ID,Chosen_BIGG,EC_numbers,EC_key,group_index,Reaction_Group
0,AAE,8268,Positive,[MNXR190571],ACS,{6.2.1.1},6.2.1.1,1,ACS_1
1,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ALATA_L,{2.6.1.1},2.6.1.1,1,ALATA_L_1
2,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ASPTA,{2.6.1.1},2.6.1.1,1,ASPTA_1
3,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",TYRTA,{2.6.1.1},2.6.1.1,1,TYRTA_1
4,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",LEUTAi,{2.6.1.1},2.6.1.1,1,LEUTAi_1


In [15]:
summary = (
    df_exploded.groupby("Reaction_Group")["Gene"]
    .apply(set)
    .reset_index(name="Genes")
)
print(summary)

           Reaction_Group                                Genes
0                13PPDH_1                         {dhaT, DhaT}
1              1P2CBXLR_1                               {dpkA}
2               1PPDCRc_1                               {dpkA}
3     1PPDCRp;1PPDCRp_1_1                               {dpkA}
4              23PDE2pp_1                               {yfkN}
...                   ...                                  ...
1618              r0668_1                               {neuA}
1619              r0737_1  {hns, yesZ, lacZ, ganA, LAC4, BgaB}
1620              r0786_1                         {BST1, DPL1}
1621              r1374_1         {NCU07035, ChiA, SCW2, chiA}
1622              r1411_1  {hns, yesZ, lacZ, ganA, LAC4, BgaB}

[1623 rows x 2 columns]


In [16]:
df_exploded.query('Chosen_BIGG=="ATPM"').query('EC_key.str.contains("2.7.1.2")')

,Gene,Paper_ID,Classification,Chosen_Reaction_ID,Chosen_BIGG,EC_numbers,EC_key,group_index,Reaction_Group
10890,dhaL,14474,Negative,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16751,nagC,8687,Negative,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16752,nagC,8687,Other,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16767,nagR,2127,Negative,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16768,nagR,2127,Other,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16769,nagR,2127,Positive,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19
16770,nagR,8824,Other,[MNXR153054],ATPM,{2.7.1.2},2.7.1.2,19,ATPM_19


In [17]:

# --- Step 1: normalize EC sets into a comparable string key ---
df_exploded["EC_key"] = df_exploded["EC_numbers"].apply(lambda s: ",".join(sorted(s)))

# --- Step 2: assign group index per reaction (ACS_1, ACS_2, etc.) ---
df_exploded["group_index"] = (
    df_exploded.groupby("Chosen_BIGG")["EC_key"]
    .transform(lambda x: pd.factorize(x)[0] + 1)
)
df_exploded["Reaction_Group"] = df_exploded["Chosen_BIGG"] + "_" + df_exploded["group_index"].astype(str)


In [18]:
df_exploded

,Gene,Paper_ID,Classification,Chosen_Reaction_ID,Chosen_BIGG,EC_numbers,EC_key,group_index,Reaction_Group
0,AAE,8268,Positive,[MNXR190571],ACS,{6.2.1.1},6.2.1.1,1,ACS_1
1,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ALATA_L,{2.6.1.1},2.6.1.1,1,ALATA_L_1
2,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",ASPTA,{2.6.1.1},2.6.1.1,1,ASPTA_1
3,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",TYRTA,{2.6.1.1},2.6.1.1,1,TYRTA_1
4,AAT1,13075,Negative,"[MNXR95698, MNXR192556, MNXR105000, MNXR192685...",LEUTAi,{2.6.1.1},2.6.1.1,1,LEUTAi_1
...,...,...,...,...,...,...,...,...,...
22710,zwf1,15355,Positive,[MNXR192434],G6PDH2r,{1.1.1.49},1.1.1.49,2,G6PDH2r_2
22711,zwf1,15789,Positive,[MNXR192434],G6PDH2r,{1.1.1.49},1.1.1.49,2,G6PDH2r_2
22712,zwf2,1120,Negative,[MNXR192434],G6PDH2r,{1.1.1.49},1.1.1.49,2,G6PDH2r_2
22713,zwf2,3017,Positive,[MNXR192434],G6PDH2r,{1.1.1.49},1.1.1.49,2,G6PDH2r_2


In [22]:

# --- Step 3: aggregate information per Reaction_Group ---
summary = (
    df_exploded.groupby(["Reaction_Group", "Chosen_BIGG", "EC_key"], dropna=False)
    .agg({
        "Gene": lambda x: set(x),
        "Paper_ID": lambda x: set(x),
        "Classification": lambda x: set(x),

        "Chosen_Reaction_ID": lambda x: set(sum(x, [])),  # flatten list of lists
        "EC_numbers": lambda x: set().union(*x)
    })
    .reset_index()
)



In [23]:
summary.head()

,Reaction_Group,Chosen_BIGG,EC_key,Gene,Paper_ID,Classification,Chosen_Reaction_ID,EC_numbers
0,13PPDH_1,13PPDH,1.1.1.202,"{dhaT, DhaT}","{6372, 2821, 2918, 9128, 10570, 7307, 4012, 42...","{Positive, Negative, Other}","{MNXR95727, MNXR94691}",{1.1.1.202}
1,1P2CBXLR_1,1P2CBXLR,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
2,1PPDCRc_1,1PPDCRc,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
3,1PPDCRp;1PPDCRp_1_1,1PPDCRp;1PPDCRp_1,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
4,23PDE2pp_1,23PDE2pp,3.1.4.16,{yfkN},{12856},{Negative},"{MNXR190422, MNXR190450, MNXR190454, MNXR19045...",{3.1.4.16}


In [24]:
summary.head()

,Reaction_Group,Chosen_BIGG,EC_key,Gene,Paper_ID,Classification,Chosen_Reaction_ID,EC_numbers
0,13PPDH_1,13PPDH,1.1.1.202,"{dhaT, DhaT}","{6372, 2821, 2918, 9128, 10570, 7307, 4012, 42...","{Positive, Negative, Other}","{MNXR95727, MNXR94691}",{1.1.1.202}
1,1P2CBXLR_1,1P2CBXLR,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
2,1PPDCRc_1,1PPDCRc,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
3,1PPDCRp;1PPDCRp_1_1,1PPDCRp;1PPDCRp_1,1.5.1.21,{dpkA},"{7704, 8273, 8945}",{Positive},"{MNXR106884, MNXR94711, MNXR94712}",{1.5.1.21}
4,23PDE2pp_1,23PDE2pp,3.1.4.16,{yfkN},{12856},{Negative},"{MNXR190422, MNXR190450, MNXR190454, MNXR19045...",{3.1.4.16}


In [25]:
# Ensure original_df has Paper_ID as index
original_df.index.name = "Paper_ID"

# Make sure Organism entries are lists (convert if some are strings or NaN)
original_df["Organism"] = original_df["Organism"].apply(
    lambda x: x if isinstance(x, (list, set)) else ([] if pd.isna(x) else [x])
)

# Function to get organisms for a set of Paper_IDs
def get_organisms(paper_ids):
    orgs = set()
    for pid in paper_ids:
        if pid in original_df.index:
            orgs.update(original_df.loc[pid, "Organism"])
    return orgs

# Apply to summary
summary["Organisms"] = summary["Paper_ID"].apply(get_organisms)

summary[["Reaction_Group", "Chosen_BIGG", "Gene", "Paper_ID", "Organisms","Classification"]]

,Reaction_Group,Chosen_BIGG,Gene,Paper_ID,Organisms,Classification
0,13PPDH_1,13PPDH,"{dhaT, DhaT}","{6372, 2821, 2918, 9128, 10570, 7307, 4012, 42...","{Yarrowia lipolytica, Shimwellia blattae, Clos...","{Positive, Negative, Other}"
1,1P2CBXLR_1,1P2CBXLR,{dpkA},"{7704, 8273, 8945}","{Pseudomonas putida, Corynebacterium glutamicum}",{Positive}
2,1PPDCRc_1,1PPDCRc,{dpkA},"{7704, 8273, 8945}","{Pseudomonas putida, Corynebacterium glutamicum}",{Positive}
3,1PPDCRp;1PPDCRp_1_1,1PPDCRp;1PPDCRp_1,{dpkA},"{7704, 8273, 8945}","{Pseudomonas putida, Corynebacterium glutamicum}",{Positive}
4,23PDE2pp_1,23PDE2pp,{yfkN},{12856},{Bacillus subtilis},{Negative}
...,...,...,...,...,...,...
1618,r0668_1,r0668,{neuA},"{14250, 16054, 103}",{Escherichia coli},{Positive}
1619,r0737_1,r0737,"{hns, yesZ, lacZ, ganA, LAC4, BgaB}","{6660, 8196, 4127, 9764, 14887, 5707, 1613, 12...","{Kluyveromyces lactis, Streptococcus thermophi...","{Positive, Negative, Other}"
1620,r0786_1,r0786,"{BST1, DPL1}","{4562, 15141}",{Saccharomyces cerevisiae},"{Positive, Negative}"
1621,r1374_1,r1374,"{NCU07035, ChiA, SCW2, chiA}","{4290, 5354, 14236, 7929}","{Bacillus licheniformis, Neurospora crassa, Se...","{Positive, Negative}"


In [26]:
summary.to_json('06_BIGG_X_EC_number_counts.json')

In [27]:
df_exploded.to_json('Correct_counts.json')